In [0]:

# Chronological regression + direction-classification bake-off.
# The production champion remains the regression model consumed by existing batch/serving jobs.

import os
import sys

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import sklearn

from mlflow.models import infer_signature
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    BaseCrossValidator,
    ParameterGrid,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from utils import (
    FEATURE_COLS,
    TARGET_COL,
    CLASS_TARGET_COL,
    chronological_cuts,
    load_model_params,
    load_project,
    recent_window_cutoff,
    regression_scores,
    widget_or,
)


class ReturnToPrice:
    """Model predicts next-session return; public predict() returns next-session close.

    shrink scales the predicted return toward 0 (the persistence/random-walk prior).
    """

    def __init__(self, base, shrink=1.0):
        self.base = base
        self.shrink = float(shrink)

    def predict(self, X):
        close = np.asarray(X["close"], dtype=float)

        return close * (
            1.0
            + self.shrink
            * np.asarray(self.base.predict(X), dtype=float)
        )


class DateGroupTimeSeriesSplit(BaseCrossValidator):
    """Expanding-window CV that never places rows from the same date in different folds."""

    def __init__(self, n_splits=4):
        self.n_splits = int(n_splits)

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError(
                "DateGroupTimeSeriesSplit requires trade_ts groups"
            )

        groups = np.asarray(groups)
        dates = np.sort(pd.unique(groups))

        if len(dates) <= self.n_splits:
            raise ValueError(
                f"Need more than {self.n_splits} distinct dates "
                f"for grouped CV; got {len(dates)}"
            )

        chunks = np.array_split(dates, self.n_splits + 1)

        for fold in range(1, len(chunks)):
            train_dates = np.concatenate(chunks[:fold])
            valid_dates = chunks[fold]

            train_idx = np.flatnonzero(
                np.isin(groups, train_dates)
            )
            valid_idx = np.flatnonzero(
                np.isin(groups, valid_dates)
            )

            if len(train_idx) and len(valid_idx):
                yield train_idx, valid_idx


def classification_scores(y_true, pred, probability):
    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, pred)
        ),
        "precision_up": float(
            precision_score(y_true, pred, zero_division=0)
        ),
        "recall_up": float(
            recall_score(y_true, pred, zero_division=0)
        ),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "up_rate_actual": float(
            np.mean(np.asarray(y_true) == 1)
        ),
        "up_rate_predicted": float(
            np.mean(np.asarray(pred) == 1)
        ),
    }

    try:
        out["roc_auc"] = float(
            roc_auc_score(y_true, probability)
        )
    except ValueError:
        out["roc_auc"] = float("nan")

    return out


# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

cfg = load_project()
m = cfg["model"]

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("selection_metric", "")
dbutils.widgets.text("search_n_iter", "")
dbutils.widgets.text("search_cv", "")
dbutils.widgets.text("val_fraction", "")
dbutils.widgets.text("test_fraction", "")
dbutils.widgets.text("max_training_rows", "")

catalog = dbutils.widgets.get("catalog")

SELECTION_METRIC = str(
    widget_or(
        dbutils.widgets.get("selection_metric"),
        m["selection_metric"],
    )
).strip()

SEARCH_N_ITER = int(
    widget_or(
        dbutils.widgets.get("search_n_iter"),
        m["search_n_iter"],
    )
)

SEARCH_CV = int(
    widget_or(
        dbutils.widgets.get("search_cv"),
        m["search_cv"],
    )
)

VAL_FRACTION = float(
    widget_or(
        dbutils.widgets.get("val_fraction"),
        m["validation_fraction"],
    )
)

TEST_FRACTION = float(
    widget_or(
        dbutils.widgets.get("test_fraction"),
        m["test_fraction"],
    )
)

MAX_ROWS = int(
    widget_or(
        dbutils.widgets.get("max_training_rows"),
        m["max_training_rows_pandas"],
    )
)

SEED = int(m["random_state"])

SELECTION_KEYS = {
    "val_rmse": ("val_rmse", True),
    "val_mae": ("val_mae", True),
    "val_r2": ("val_r2", False),
    "val_directional_accuracy": (
        "val_directional_accuracy",
        False,
    ),
}

if SELECTION_METRIC not in SELECTION_KEYS:
    raise ValueError(
        f"selection_metric must be one of {list(SELECTION_KEYS)}"
    )


# ---------------------------------------------------------
# 2. MLFLOW CONFIGURATION
# ---------------------------------------------------------

mlflow.set_experiment(m["experiment_path"])
mlflow.autolog(log_models=False)


# ---------------------------------------------------------
# 3. LOAD FEATURE DATA
# ---------------------------------------------------------

sdf = spark.table(
    f"{catalog}.{cfg['tables']['features']}"
).select(
    "symbol",
    "trade_ts",
    *FEATURE_COLS,
    TARGET_COL,
    CLASS_TARGET_COL,
)

ts_counts = sdf.groupBy("trade_ts").count().toPandas()

n_rows = int(ts_counts["count"].sum())

keep_from = recent_window_cutoff(
    zip(ts_counts["trade_ts"], ts_counts["count"]),
    MAX_ROWS,
)

if keep_from is not None:
    sdf = sdf.filter(
        sdf.trade_ts >= pd.Timestamp(keep_from).to_pydatetime()
    )

    print(
        f"Training on recent complete-date window from {keep_from}; "
        f"source rows={n_rows:,}"
    )

pdf = sdf.orderBy(
    "trade_ts",
    "symbol",
).toPandas()

pdf = pdf.replace(
    [np.inf, -np.inf],
    np.nan,
)

pdf = pdf.dropna(
    subset=FEATURE_COLS + [TARGET_COL, CLASS_TARGET_COL]
)

pdf[FEATURE_COLS] = pdf[FEATURE_COLS].astype("float64")

pdf[CLASS_TARGET_COL] = pdf[CLASS_TARGET_COL].astype("int64")


# ---------------------------------------------------------
# 4. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ---------------------------------------------------------

cut1, cut2 = chronological_cuts(
    pdf.trade_ts.unique(),
    VAL_FRACTION,
    TEST_FRACTION,
)

fit_df = pdf[pdf.trade_ts <= cut1].copy()

val_df = pdf[
    (pdf.trade_ts > cut1)
    & (pdf.trade_ts <= cut2)
].copy()

test_df = pdf[pdf.trade_ts > cut2].copy()

if min(map(len, (fit_df, val_df, test_df))) == 0:
    raise ValueError(
        "Chronological split produced an empty segment"
    )

X_fit = fit_df[FEATURE_COLS]
X_val = val_df[FEATURE_COLS]
X_test = test_df[FEATURE_COLS]

y_fit_price = fit_df[TARGET_COL]
y_val_price = val_df[TARGET_COL]
y_test_price = test_df[TARGET_COL]

y_fit_ret = (
    y_fit_price / X_fit["close"] - 1.0
)

groups_fit = fit_df["trade_ts"].to_numpy()

cv = DateGroupTimeSeriesSplit(
    n_splits=SEARCH_CV
)

print(
    f"Rows fit/val/test: "
    f"{len(fit_df):,}/{len(val_df):,}/{len(test_df):,}; "
    f"dates: "
    f"{fit_df.trade_ts.nunique()}/"
    f"{val_df.trade_ts.nunique()}/"
    f"{test_df.trade_ts.nunique()}; "
    f"symbols={pdf.symbol.nunique()}"
)


# ---------------------------------------------------------
# 5. REGRESSION MODEL FACTORIES
# ---------------------------------------------------------

param_grids = load_model_params()

MODEL_FACTORIES = {
    "elastic_net": lambda: Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "reg",
                ElasticNet(
                    random_state=SEED,
                    max_iter=10000,
                ),
            ),
        ]
    ),

    "random_forest": lambda: RandomForestRegressor(
        n_jobs=-1,
        random_state=SEED,
    ),

    "extra_trees": lambda: ExtraTreesRegressor(
        n_jobs=-1,
        random_state=SEED,
    ),

    "hist_gradient_boosting": lambda: (
        HistGradientBoostingRegressor(
            random_state=SEED,
        )
    ),

    "gradient_boosting": lambda: (
        GradientBoostingRegressor(
            random_state=SEED,
        )
    ),
}


# Optional XGBoost
try:
    from xgboost import XGBRegressor

    MODEL_FACTORIES["xgboost"] = lambda: XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=SEED,
        verbosity=0,
    )

except ImportError:
    print("Optional model xgboost not installed; skipping")


# Optional LightGBM
try:
    from lightgbm import LGBMRegressor

    MODEL_FACTORIES["lightgbm"] = lambda: LGBMRegressor(
        n_jobs=-1,
        random_state=SEED,
        verbosity=-1,
    )

except ImportError:
    print("Optional model lightgbm not installed; skipping")


# Optional CatBoost
try:
    from catboost import CatBoostRegressor

    MODEL_FACTORIES["catboost"] = lambda: CatBoostRegressor(
        verbose=False,
        random_seed=SEED,
        thread_count=-1,
        loss_function="RMSE",
    )

except ImportError:
    print("Optional model catboost not installed; skipping")


unknown = [
    model_name
    for model_name in param_grids
    if model_name not in MODEL_FACTORIES
]

if unknown:
    print(
        f"Models not available in this runtime "
        f"(or unknown): {unknown}"
    )

models_to_run = [
    model_name
    for model_name in param_grids
    if model_name in MODEL_FACTORIES
]

if not models_to_run:
    raise ValueError(
        "No supported regression model blocks found "
        "in conf/model_params.yml"
    )


# ---------------------------------------------------------
# 6. HELPER FUNCTIONS
# ---------------------------------------------------------

def search_space(model, grid):
    if isinstance(model, Pipeline):
        return {
            f"reg__{key}": value
            for key, value in grid.items()
        }

    return grid


def per_symbol_metrics(frame, pred):
    rows = []

    for sym, group in frame.assign(
        _pred=pred
    ).groupby("symbol"):

        rows.append(
            {
                "symbol": sym,
                "rows": len(group),
                **regression_scores(
                    group[TARGET_COL],
                    group["_pred"],
                    group["close"],
                ),
            }
        )

    return pd.DataFrame(rows)


# ---------------------------------------------------------
# 7. REGRESSION + CLASSIFICATION BAKE-OFF
# ---------------------------------------------------------

regression_results = []
classification_results = []

with mlflow.start_run(
    run_name="stock_regression_and_direction_bakeoff"
) as parent_run:

    mlflow.log_params(
        {
            "selection_metric": SELECTION_METRIC,
            "search_n_iter": SEARCH_N_ITER,
            "search_cv": SEARCH_CV,
            "cv_strategy": "expanding_date_grouped",
            "n_symbols": int(pdf.symbol.nunique()),
            "source_rows": n_rows,
            "training_rows": int(len(pdf)),
            "training_window_start": str(
                keep_from or "full_history"
            ),
            "validation_fraction": VAL_FRACTION,
            "test_fraction": TEST_FRACTION,
            "feature_count": len(FEATURE_COLS),
        }
    )

    # -----------------------------------------------------
    # 7A. REGRESSION BAKE-OFF
    # -----------------------------------------------------

    for name in models_to_run:

        model = MODEL_FACTORIES[name]()

        with mlflow.start_run(
            run_name=f"regression_{name}",
            nested=True,
        ) as child:

            space = search_space(
                model,
                param_grids[name],
            )

            n_iter = min(
                SEARCH_N_ITER,
                len(ParameterGrid(space)),
            )

            search = RandomizedSearchCV(
                model,
                space,
                n_iter=n_iter,
                scoring="neg_root_mean_squared_error",
                cv=cv,
                random_state=SEED,
                n_jobs=-1,
                refit=True,
                error_score="raise",
            )

            search.fit(
                X_fit,
                y_fit_ret,
                groups=groups_fit,
            )

            final_base = (
                clone(model)
                .set_params(**search.best_params_)
                .fit(X_fit, y_fit_ret)
            )

            # Calibrate shrinkage using validation data only.
            # Test data is never used to select the shrink factor.

            r_hat_val = np.asarray(
                final_base.predict(X_val),
                dtype=float,
            )

            close_val = np.asarray(
                X_val["close"],
                dtype=float,
            )

            r_val = (
                np.asarray(y_val_price, dtype=float)
                / close_val
                - 1.0
            )

            per_se = float(
                np.mean((close_val * r_val) ** 2)
            )

            if per_se <= 0 or not np.isfinite(per_se):
                raise ValueError(
                    "Invalid persistence baseline error "
                    "during shrinkage calibration"
                )

            shrink_candidates = np.round(
                np.arange(0.0, 1.05, 0.05),
                2,
            )

            shrink = min(
                shrink_candidates,
                key=lambda shrink_value: (
                    float(
                        np.mean(
                            (
                                close_val
                                * (
                                    r_val
                                    - shrink_value * r_hat_val
                                )
                            ) ** 2
                        )
                    ) / per_se
                ),
            )

            final_model = ReturnToPrice(
                final_base,
                shrink=shrink,
            )

            val_pred = final_model.predict(X_val)
            test_pred = final_model.predict(X_test)

            val_metrics = regression_scores(
                y_val_price,
                val_pred,
                val_df.close,
            )

            test_metrics = regression_scores(
                y_test_price,
                test_pred,
                test_df.close,
            )

            mlflow.log_params(
                {
                    "algorithm": name,
                    "persistence_shrinkage": float(shrink),
                    **{
                        key.replace("reg__", ""): value
                        for key, value in search.best_params_.items()
                    },
                }
            )

            mlflow.set_tag(
                "task_type",
                "regression",
            )

            mlflow.log_metric(
                "cv_best_rmse_return",
                float(-search.best_score_),
            )

            for prefix, values in (
                ("val_", val_metrics),
                ("test_", test_metrics),
            ):

                mlflow.log_metrics(
                    {
                        prefix + key: float(value)
                        for key, value in values.items()
                    }
                )

            mlflow.log_text(
                per_symbol_metrics(
                    test_df,
                    test_pred,
                ).to_csv(index=False),
                "per_symbol_test_metrics.csv",
            )

            signature = infer_signature(
                X_fit.head(1000),
                final_model.predict(X_fit.head(1000)),
            )

            mlflow.sklearn.log_model(
                final_model,
                artifact_path="model",
                signature=signature,
                input_example=X_fit.head(5),
                extra_pip_requirements=[
                    f"scikit-learn=={sklearn.__version__}",
                    "numpy<2.0",
                ],
            )

            regression_results.append(
                {
                    "algorithm": name,
                    "run_id": child.info.run_id,
                    **{
                        "val_" + key: value
                        for key, value in val_metrics.items()
                    },
                    **{
                        "test_" + key: value
                        for key, value in test_metrics.items()
                    },
                    "cv_best_rmse_return": float(
                        -search.best_score_
                    ),
                }
            )

            print(
                f"REG {name}: "
                f"val RMSE={val_metrics['rmse']:.5f}; "
                f"val direction="
                f"{val_metrics['directional_accuracy']:.3f}; "
                f"test RMSE={test_metrics['rmse']:.5f}; "
                f"test direction="
                f"{test_metrics['directional_accuracy']:.3f}; "
                f"shrink={shrink:.2f}"
            )

    # -----------------------------------------------------
    # 7B. DIRECTION CLASSIFICATION BAKE-OFF
    # -----------------------------------------------------

    # This classifier is separate from the existing
    # production price-regression champion.

    classifier_factories = {

        "logistic_regression": lambda: Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=SEED,
                    ),
                ),
            ]
        ),

        "random_forest_classifier": lambda: (
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=SEED,
            )
        ),

        "extra_trees_classifier": lambda: (
            ExtraTreesClassifier(
                n_estimators=300,
                min_samples_leaf=10,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,
            )
        ),

        "hist_gradient_classifier": lambda: (
            HistGradientBoostingClassifier(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=SEED,
            )
        ),

        "gradient_boosting_classifier": lambda: (
            GradientBoostingClassifier(
                n_estimators=150,
                learning_rate=0.03,
                max_depth=2,
                random_state=SEED,
            )
        ),
    }

    y_fit_cls = fit_df[CLASS_TARGET_COL].to_numpy()
    y_val_cls = val_df[CLASS_TARGET_COL].to_numpy()
    y_test_cls = test_df[CLASS_TARGET_COL].to_numpy()

    classifier_spaces = {

        "logistic_regression": {
            "clf__C": [0.01, 0.1, 1.0, 3.0, 10.0],
        },

        "random_forest_classifier": {
            "n_estimators": [150, 300, 500],
            "max_depth": [4, 8, 12, None],
            "min_samples_leaf": [5, 10, 25],
            "max_features": [0.5, 0.75, 1.0],
        },

        "extra_trees_classifier": {
            "n_estimators": [150, 300, 500],
            "max_depth": [4, 8, 12, None],
            "min_samples_leaf": [5, 10, 25],
            "max_features": [0.5, 0.75, 1.0],
        },

        "hist_gradient_classifier": {
            "max_iter": [100, 200, 300],
            "learning_rate": [0.02, 0.05, 0.1],
            "max_leaf_nodes": [7, 15, 31],
            "l2_regularization": [0.0, 0.5, 2.0],
        },

        "gradient_boosting_classifier": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.02, 0.05, 0.1],
            "max_depth": [1, 2, 3],
            "min_samples_leaf": [10, 25, 50],
            "subsample": [0.7, 0.85, 1.0],
        },
    }

    for name, factory in classifier_factories.items():

        model = factory()

        with mlflow.start_run(
            run_name=f"classification_{name}",
            nested=True,
        ) as child:

            cls_space = classifier_spaces[name]

            cls_search = RandomizedSearchCV(
                model,
                cls_space,
                n_iter=min(
                    SEARCH_N_ITER,
                    len(ParameterGrid(cls_space)),
                ),
                scoring="balanced_accuracy",
                cv=cv,
                random_state=SEED,
                n_jobs=-1,
                refit=True,
                error_score="raise",
            )

            cls_search.fit(
                X_fit,
                y_fit_cls,
                groups=groups_fit,
            )

            model = cls_search.best_estimator_

            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)

            val_prob = (
                model.predict_proba(X_val)[:, 1]
                if hasattr(model, "predict_proba")
                else val_pred.astype(float)
            )

            test_prob = (
                model.predict_proba(X_test)[:, 1]
                if hasattr(model, "predict_proba")
                else test_pred.astype(float)
            )

            val_metrics = classification_scores(
                y_val_cls,
                val_pred,
                val_prob,
            )

            test_metrics = classification_scores(
                y_test_cls,
                test_pred,
                test_prob,
            )

            mlflow.set_tag(
                "task_type",
                "direction_classification",
            )

            mlflow.set_tag(
                "algorithm",
                name,
            )

            mlflow.log_params(
                {
                    "algorithm": name,
                    "target": CLASS_TARGET_COL,
                    "cv_best_balanced_accuracy": float(
                        cls_search.best_score_
                    ),
                    **cls_search.best_params_,
                }
            )

            cls_mlflow_metrics = {
                "val_" + key: value
                for key, value in val_metrics.items()
            }

            cls_mlflow_metrics.update(
                {
                    "test_" + key: value
                    for key, value in test_metrics.items()
                }
            )

            mlflow.log_metrics(
                {
                    key: float(value)
                    for key, value in cls_mlflow_metrics.items()
                    if np.isfinite(value)
                }
            )

            signature = infer_signature(
                X_fit.head(1000),
                model.predict(X_fit.head(1000)),
            )

            mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                signature=signature,
                input_example=X_fit.head(5),
                extra_pip_requirements=[
                    f"scikit-learn=={sklearn.__version__}",
                ],
            )

            classification_results.append(
                {
                    "algorithm": name,
                    "run_id": child.info.run_id,
                    **{
                        "val_" + key: value
                        for key, value in val_metrics.items()
                    },
                    **{
                        "test_" + key: value
                        for key, value in test_metrics.items()
                    },
                }
            )

            print(
                f"CLS {name}: "
                f"val balanced acc="
                f"{val_metrics['balanced_accuracy']:.3f}; "
                f"test accuracy="
                f"{test_metrics['accuracy']:.3f}; "
                f"test balanced acc="
                f"{test_metrics['balanced_accuracy']:.3f}; "
                f"test ROC-AUC="
                f"{test_metrics['roc_auc']:.3f}"
            )

    # -----------------------------------------------------
    # 8. SELECT BEST REGRESSION AND CLASSIFICATION MODELS
    # -----------------------------------------------------

    # Production price-regression model is selected using
    # validation metrics only.

    sel_key, lower_is_better = SELECTION_KEYS[
        SELECTION_METRIC
    ]

    regression_summary = pd.DataFrame(
        regression_results
    )

    best = (
        regression_summary
        .sort_values(
            sel_key,
            ascending=lower_is_better,
        )
        .iloc[0]
        .to_dict()
    )

    classifier_summary = (
        pd.DataFrame(classification_results)
        .sort_values(
            "val_balanced_accuracy",
            ascending=False,
        )
    )

    best_classifier = (
        classifier_summary
        .iloc[0]
        .to_dict()
    )

    mlflow.log_params(
        {
            "best_regression_algorithm": best["algorithm"],
            "best_direction_classifier": (
                best_classifier["algorithm"]
            ),
        }
    )

    mlflow.log_metrics(
        {
            "winning_validation_rmse": float(
                best["val_rmse"]
            ),
            "best_classifier_validation_balanced_accuracy": (
                float(
                    best_classifier[
                        "val_balanced_accuracy"
                    ]
                )
            ),
        }
    )

    mlflow.set_tag(
        "best_regression_run_id",
        best["run_id"],
    )

    mlflow.set_tag(
        "best_classifier_run_id",
        best_classifier["run_id"],
    )

    mlflow.log_text(
        regression_summary.to_csv(index=False),
        "regression_model_comparison.csv",
    )

    mlflow.log_text(
        classifier_summary.to_csv(index=False),
        "direction_classifier_comparison.csv",
    )

    run_id = best["run_id"]
    parent_run_id = parent_run.info.run_id


# ---------------------------------------------------------
# 9. DISPLAY MODEL COMPARISON RESULTS
# ---------------------------------------------------------

print(
    "\nREGRESSION COMPARISON "
    "(sorted by validation selection metric)"
)

print(
    regression_summary
    .sort_values(
        sel_key,
        ascending=lower_is_better,
    )
    .round(4)
    .to_string(index=False)
)

print(
    "\nDIRECTION CLASSIFICATION COMPARISON "
    "(sorted by validation balanced accuracy)"
)

print(
    classifier_summary
    .round(4)
    .to_string(index=False)
)


# ---------------------------------------------------------
# 10. PUBLISH DATABRICKS TASK VALUES
# ---------------------------------------------------------

# Downstream tasks continue to consume the best
# regression run exactly as before.

dbutils.jobs.taskValues.set(
    key="run_id",
    value=run_id,
)

dbutils.jobs.taskValues.set(
    key="parent_run_id",
    value=parent_run_id,
)

dbutils.jobs.taskValues.set(
    key="best_algorithm",
    value=best["algorithm"],
)

dbutils.jobs.taskValues.set(
    key="validation_rmse",
    value=float(best["val_rmse"]),
)

dbutils.jobs.taskValues.set(
    key="test_rmse",
    value=float(best["test_rmse"]),
)

dbutils.jobs.taskValues.set(
    key="test_directional_accuracy",
    value=float(best["test_directional_accuracy"]),
)

dbutils.jobs.taskValues.set(
    key="best_classifier_algorithm",
    value=best_classifier["algorithm"],
)

dbutils.jobs.taskValues.set(
    key="best_classifier_run_id",
    value=best_classifier["run_id"],
)

dbutils.jobs.taskValues.set(
    key="classifier_test_accuracy",
    value=float(best_classifier["test_accuracy"]),
)

dbutils.jobs.taskValues.set(
    key="classifier_test_balanced_accuracy",
    value=float(
        best_classifier["test_balanced_accuracy"]
    ),
)


# ---------------------------------------------------------
# 11. FINAL OUTPUT
# ---------------------------------------------------------

print(
    f"\nProduction regression champion candidate: "
    f"{best['algorithm']} | run={run_id}"
)

print(
    f"Best direction classifier: "
    f"{best_classifier['algorithm']} | "
    f"run={best_classifier['run_id']}"
)
